# C2: Timeline Analysis

---

## Learning Objectives

By the end of this notebook, you will be able to:
1. **Calculate processing times** between permit stages
2. **Identify bottlenecks** in the approval pipeline
3. **Compare trends** year-over-year
4. **Visualize timelines** for policy insights

## Why This Matters

Time is money in housing development:
- Every month of delay adds costs (~1-2% of project value)
- Long review times discourage builders
- Identifying bottlenecks helps prioritize reforms

This analysis answers: "Where do projects get stuck, and for how long?"

## Key Metrics

| Metric | What It Shows |
|--------|---------------|
| Median review time | Typical processing duration |
| 90th percentile | Worst-case scenarios |
| Status age | How long projects sit in each stage |
| Completion rate | % finishing within expected time |

## NEW: Stage-by-Stage Timeline from permit_events

With our new `permit_events` table, we can now calculate **exact durations** between stages:

```
Completeness Review → CEQA Determination → Staff Decision → Appeal → Issuance → CO
        (X days)           (Y days)           (Z days)      (if any)
```

**Dual timestamps:** Each event has `event_date` (when it happened) and `imported_at` (when we recorded it).

---

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Find project root and setup environment
def find_project_root():
    """Find project root by looking for marker directories."""
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if (path / '00_config').exists() and (path / 'modules').exists():
            return path
    raise FileNotFoundError("Could not find project root")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

# Load config with resolved paths
import json
with open(ROOT / '00_config/berkeley_config.json') as f:
    CONFIG = json.load(f)

# Resolve relative paths to absolute
for key, value in CONFIG['paths'].items():
    if isinstance(value, str) and not value.startswith('http'):
        CONFIG['paths'][key] = str(ROOT / value)

print(f"✅ Project root: {ROOT}")
print(f"✅ Housing data: {CONFIG['paths']['housing_projects']}")

## 1a. Stage-by-Stage Duration Analysis (NEW)

Query the `permit_events` table to calculate exact days at each pipeline stage.

In [ ]:
import sqlite3
import pandas as pd

# Connect to database
DB_PATH = ROOT / CONFIG['paths']['database']

def analyze_stage_durations(db_path):
    """
    Analyze time spent at each permit stage.
    
    Uses dual timestamp system:
    - event_date: when the action occurred (for timeline analysis)
    - imported_at: when we recorded it (for data quality)
    """
    conn = sqlite3.connect(db_path)
    
    # Query stage durations for each project
    df = pd.read_sql_query("""
        SELECT 
            p.address_display,
            p.net_units,
            pe.permit_number,
            pe.stage,
            pe.action,
            pe.event_date,
            pe.marked_by,
            LAG(pe.event_date) OVER (
                PARTITION BY pe.project_id 
                ORDER BY pe.event_date
            ) as prev_event_date
        FROM permit_events pe
        JOIN projects p ON pe.project_id = p.id
        ORDER BY p.net_units DESC, pe.event_date
    """, conn)
    
    conn.close()
    
    # Calculate days between events
    df['event_date'] = pd.to_datetime(df['event_date'])
    df['prev_event_date'] = pd.to_datetime(df['prev_event_date'])
    df['days_since_last'] = (df['event_date'] - df['prev_event_date']).dt.days
    
    return df

# Run analysis
df_stages = analyze_stage_durations(DB_PATH)

if len(df_stages) > 0:
    print(f"Analyzed {len(df_stages)} permit events")
    
    # Summarize by stage
    stage_summary = df_stages.groupby('stage').agg({
        'days_since_last': ['mean', 'median', 'max', 'count']
    }).round(0)
    stage_summary.columns = ['Avg Days', 'Median Days', 'Max Days', 'Events']
    stage_summary = stage_summary.sort_values('Median Days', ascending=False)
    
    print("\nTime Spent by Stage:")
    print("="*60)
    display(stage_summary)
else:
    print("No permit events found. Run Accela data collection workflow first.")

## 2. Load Data

In [ ]:
# Load housing projects
housing_path = Path(CONFIG['paths']['housing_projects'])
df = load_csv(housing_path)

if df is not None:
    print(f"Loaded {len(df)} projects")
    print(f"\nColumns available: {df.columns.tolist()}")

## 3. Year-Based Analysis

Since we have year data, analyze by filing year.

In [ ]:
# Distribution by year
if df is not None and 'year' in df.columns:
    year_summary = df.groupby('year').agg({
        'address_display': 'count',
        'net_units': ['sum', 'mean']
    }).round(1)
    
    year_summary.columns = ['Projects', 'Total Units', 'Avg Units']
    year_summary = year_summary.sort_index(ascending=False)
    
    print("Projects by Year:")
    display(year_summary)

## 4. Timeline by Project Size

In [ ]:
# Analysis by project size
if df is not None and 'project_size_category' in df.columns:
    size_summary = df.groupby('project_size_category').agg({
        'address_display': 'count',
        'net_units': 'sum',
        'status': lambda x: x.value_counts().index[0]  # Most common status
    }).reset_index()
    
    size_summary.columns = ['Size Category', 'Projects', 'Total Units', 'Most Common Status']
    
    print("Analysis by Project Size:")
    display(size_summary)

## 5. Status Duration Estimates

Estimate typical duration at each status based on year filed.

In [ ]:
# Calculate estimated age of projects
if df is not None and 'year' in df.columns:
    current_year = 2025
    df['years_in_pipeline'] = current_year - df['year']
    
    # Average years by status
    status_age = df.groupby('status').agg({
        'years_in_pipeline': ['mean', 'min', 'max', 'count']
    }).round(1)
    
    status_age.columns = ['Avg Years', 'Min Years', 'Max Years', 'Projects']
    status_age = status_age.sort_values('Avg Years', ascending=False)
    
    print("Time in Pipeline by Current Status:")
    display(status_age)

## 6. Bottleneck Identification

In [ ]:
# Identify bottlenecks (statuses with high project counts and long durations)
if df is not None:
    print("Potential Bottlenecks (high count, long duration):")
    print("="*60)
    
    # Statuses with most projects
    status_counts = df['status'].value_counts()
    
    for status, count in status_counts.head(5).items():
        units = df[df['status'] == status]['net_units'].sum()
        avg_years = df[df['status'] == status]['years_in_pipeline'].mean() if 'years_in_pipeline' in df.columns else 0
        print(f"\n{status}:")
        print(f"  Projects: {count}")
        print(f"  Units: {units:,.0f}")
        print(f"  Avg years in pipeline: {avg_years:.1f}")

## 7. Seasonal Patterns

Note: Requires date-level data for true seasonal analysis.

In [ ]:
# Year-over-year growth
if df is not None and 'year' in df.columns:
    yearly = df.groupby('year')['net_units'].sum().sort_index()
    yearly_growth = yearly.pct_change() * 100
    
    growth_df = pd.DataFrame({
        'Year': yearly.index,
        'Units': yearly.values,
        'YoY Growth %': yearly_growth.values
    }).dropna()
    
    print("Year-over-Year Growth:")
    display(growth_df.round(1))

## 8. Export Analysis

In [ ]:
# Export timeline analysis
if df is not None:
    output_path = DATA_DIR / 'timeline_analysis.csv'
    
    analysis_cols = ['address_display', 'net_units', 'status', 'year']
    if 'years_in_pipeline' in df.columns:
        analysis_cols.append('years_in_pipeline')
    
    df[analysis_cols].to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

---

## Summary

This notebook analyzed:
- Project distribution by year
- Timeline by project size
- Duration estimates by status
- Potential bottlenecks
- Year-over-year growth

**Next:** Run `C3_proposal_vs_reality.ipynb` to compare proposed vs actual.